# Kahneman Framing × TRIBE v2 — Quick Colab Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akifnu/DSprojects/blob/main/tribev2/notebooks/Kahneman_Framing_RCT.ipynb)

**3 classic pairs · 6 texts · text-only · no audio**

1. **Runtime → GPU** (A100 best; T4 may work in quick mode)
2. Colab **Secrets** → `HF_TOKEN` (Hugging Face read token with LLaMA access)
3. **Runtime → Run all** — first run installs deps and restarts once automatically

Typical runtime after the first install: **~10–20 min** on A100 (model download is one-time per session).

In [ ]:
import os, subprocess, sys

MARKER = '/content/.tribev2_colab_ready_v6'
REPO_DIR = '/content/DSprojects'
REPO_URL = 'https://github.com/akifnu/DSprojects.git'

if not os.path.exists(MARKER):
    if not os.path.exists(REPO_DIR):
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
    req = f'{REPO_DIR}/tribev2/requirements-colab.txt'
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}/tribev2'])
    open(MARKER, 'w').write('ok')
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)

print('Dependencies ready — running demo...')

In [ ]:
import sys
sys.path.insert(0, '/content/DSprojects/tribev2/src')

import pandas as pd
from scipy import stats
from tribe_capabilities.colab import run_quick_demo

demo = run_quick_demo()

if not demo.rows:
    raise RuntimeError(
        'No completed pairs. Check errors below, restart runtime, and Run all again.\n'
        + str(demo.errors)
    )

df = pd.DataFrame(demo.rows)
display(df[['id', 'domain', 'gain_mean_abs', 'loss_mean_abs', 'loss_minus_gain']])

gain_vals = df['gain_mean_abs'].values
loss_vals = df['loss_mean_abs'].values
diff = loss_vals - gain_vals
t_stat, p_val = stats.ttest_rel(loss_vals, gain_vals)
cohens_dz = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else float('nan')

print('\n--- Kahneman framing (3-pair quick demo) ---')
print(f'Pairs: {len(df)} | Loss > gain: {(diff > 0).sum()}/{len(df)}')
print(f'Mean diff (loss - gain): {diff.mean():.4f} | p = {p_val:.4f} | dz = {cohens_dz:.3f}')
print(f'Checkpoint: {demo.checkpoint_path}')
if demo.errors:
    print('Non-fatal errors:', demo.errors)